# Build a Local Revenue Report

In this notebook, you will turn raw taxi rides into a reusable monthly revenue report. It connects the earlier Polars and DuckDB ideas to a small reporting workflow that also exists as a command-line script in `src/revenue_report.py`.

**Learning goals:**
- Turn raw Parquet files into a reusable revenue report.
- Write the result back to Parquet.
- Understand how the notebook relates to `src/revenue_report.py`.

Before you run this notebook, make sure local Parquet files are available in `data/raw/`.


## Outline

1. Define the input and output paths.
2. Build the reporting query.
3. Write the report to Parquet.
4. Compare the notebook workflow with the CLI script.


In [ ]:
from pathlib import Path

import duckdb
import polars as pl
import pyarrow.parquet as pq

INPUT_GLOB = "../data/raw/yellow_tripdata_2025-*.parquet"
OUTPUT_PATH = Path("../data/output/revenue_report_from_notebook.parquet")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

if not list(Path("../data/raw").glob("yellow_tripdata_2025-*.parquet")):
    raise FileNotFoundError("Download local data first with src/data_ingestion.py.")

con = duckdb.connect()

## Step 1 - Define the SQL once

A clear reporting query is easy to test and easy to reuse in a script later.


In [ ]:
REPORT_SQL = """
WITH trips_data AS (
    SELECT
        PULocationID,
        CAST(tpep_pickup_datetime AS TIMESTAMP) AS pickup_datetime,
        passenger_count,
        trip_distance,
        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        total_amount,
        congestion_surcharge
    FROM read_parquet(?)
)
SELECT
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month,
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM trips_data
GROUP BY 1, 2
ORDER BY revenue_month, revenue_zone
"""

pl.from_arrow(con.execute(REPORT_SQL, [INPUT_GLOB]).to_arrow_table()).head()

## Step 2 - Write the report to Parquet

The reporting query is now stable enough to materialize. Writing to Parquet keeps the result easy to share with another notebook or script.


In [ ]:
report_table = con.execute(REPORT_SQL, [INPUT_GLOB]).to_arrow_table()
pq.write_table(report_table, OUTPUT_PATH)

{"rows_written": report_table.num_rows, "output_path": str(OUTPUT_PATH)}

## Step 3 - Read the finished report

A good batch workflow usually ends by storing a smaller, more useful table than the original raw dataset.


In [ ]:
pl.from_arrow(
    con.execute(
        """
        SELECT *
        FROM read_parquet(?)
        ORDER BY revenue_month, revenue_zone
        LIMIT 10
        """,
        [str(OUTPUT_PATH)],
    ).to_arrow_table()
)

## Exercises

- Change the grouping so the report is monthly by `VendorID` instead of `PULocationID`.
- Also add a column for the maximum `total_amount` in each group.
- Run the CLI script and compare its output path with the notebook output.


In [ ]:
# TODO: Write your solutions for the exercises above.

command = (
    "uv run python src/revenue_report.py "
    '--input "data/raw/yellow_tripdata_2025-*.parquet" '
    "--output data/output/revenue_report.parquet"
)

print(command)

<details>
    <summary>Show Reference Solution</summary>

```python
REPORT_SQL_BY_VENDOR = """
WITH trips_data AS (
    SELECT
        VendorID,
        CAST(tpep_pickup_datetime AS TIMESTAMP) AS pickup_datetime,
        passenger_count,
        trip_distance,
        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        total_amount,
        congestion_surcharge
    FROM read_parquet(?)
)
SELECT
    VendorID AS revenue_vendor,
    date_trunc('month', pickup_datetime) AS revenue_month,
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance,
    MAX(total_amount) AS max_trip_amount
FROM trips_data
GROUP BY 1, 2
ORDER BY revenue_month, revenue_vendor
"""

pl.from_arrow(con.execute(REPORT_SQL_BY_VENDOR, [INPUT_GLOB]).to_arrow_table()).head(10)

```
</details>
